# Query/Gallery PQ 조건을 포함한 통합 Quick/Full 실행기

이 노트북은 기존 데이터셋별 노트북을 대체하지 않고, 동일한 canonical `research/` 단계 함수를 한 곳에서 순차 호출하는 상위 실행기입니다.

- 기본값은 `full`이며 LFW, SurvFace, RFW-Custom, TinyFace 전체 표본을 선택합니다.
- `quick`은 LFW 10%, SurvFace 2%, RFW-Custom 10%, TinyFace 10%의 identity/role-preserving 표본을 사용합니다.
- `DATASET_IDS`, `QUICK_DATA_FRACTIONS`, `COMPLETED_RUN_OVERRIDES`는 네 데이터셋을 하나의 계약으로 관리합니다.
- `arc`, `ada`, `mag`, `edge` 중 한 pretrained checkpoint를 선택합니다. 네 모델 비교는 모델별 독립 run을 반복하며 비교 단위는 checkpoint입니다.
- 내부 dispatcher는 LFW/SurvFace/RFW-Custom을 open-set Step4로, TinyFace를 공식 closed-set distractor protocol로 실행합니다.
- open-set run은 Origin, PCA direct/reconstruction, PQ reconstruction/ADC, Grad-CAM, saliency×compression, compact artifact를 유지합니다.
- TinyFace는 mAP와 Rank-1/5/10/20을 산출하며 non-mated probe가 없으므로 FPIR/TPIR calibration에는 포함하지 않습니다.
- 실제 장시간 실험은 `EXECUTE=True`와 실행 확인 값이 모두 설정된 경우에만 시작됩니다.
- Faiss IVF-PQ, pgvector IVFFlat, ANN sweep, BalancedFace, uncertainty/defer 실험은 이번 구현 범위에서 유예합니다.

## PQ paper comparison contract

파생 v6 search-space는 Origin, 양쪽 PQ reconstruction cosine,
원본 query + 복원 gallery one-sided cosine, ADC, SDC를 분리합니다.
cosine score space만 frozen-origin 진단을 허용하며 ADC/SDC는 각 score
space의 calibration threshold만 사용합니다.


## 1. 사용자가 조절하는 변수

`DATASET_IDS`, `RUN_TIER`, `MODEL_NAME`을 선택합니다. 기본 `RUN_TIER="full"`은 선택된 모든 데이터셋에서 항상 100%를 사용합니다. `RUN_TIER="quick"`일 때만 `QUICK_DATA_FRACTIONS`의 데이터셋별 비율이 적용됩니다.

`MODEL_PROFILE_BY_NAME`과 `MODEL_WEIGHT_PATHS`는 같은 학습 데이터·아키텍처의 checkpoint여야 합니다. `START_NEW_RUN=True`는 동일 plan의 완료 run이 있어도 독립 재실험을 의도할 때만 사용합니다. 완료 run 재사용은 네 데이터셋 모두 `COMPLETED_RUN_OVERRIDES`에 명시합니다.

RFW-Official 1:1 평가는 `RUN_RFW_VERIFICATION`으로 별도 활성화합니다. 이는 RFW-Custom open-set 및 TinyFace official closed-set 결과와 합치지 않습니다.


In [1]:
from __future__ import annotations

# 네 데이터셋은 하나의 선택·비율·완료-run override 계약을 사용합니다.
DATASET_IDS = ("lfw", "survface", "rfw_custom", "tinyface")
# DATASET_IDS = ("tinyface", )
RUN_TIER = "full"  # 기본값: "full"; 실행 경로 점검 때만 "quick"
# RUN_TIER = "quick"  # 기본값: "full"; 실행 경로 점검 때만 "quick"
QUICK_DATA_FRACTIONS = {
    "lfw": 0.10,
    "survface": 0.02,
    "rfw_custom": 0.10,
    "tinyface": 0.10,
}

TARGET_FPIRS = (0.01, 0.05, 0.10, 0.20, 0.30)  # open-set TPIR20 curve contract
EXPECTED_PQ_SEARCH_MODES = (
    "pq_reconstruction_cosine",
    "pq_one_sided_cosine",
    "pq_adc_exhaustive",
    "pq_sdc_exhaustive",
)
RUN_PQ_SDC = False  # True: PQ-m128 SDC 보조 ablation 실행; False: SDC 전체 제외
PQ_SDC_SETTINGS = ((128, 8),) if RUN_PQ_SDC else ()
SEED = 8972

MODEL_NAME = "arc"  # "arc", "ada", "mag", "edge" 중 하나
MODEL_PROFILE_BY_NAME = {
    "arc": "arcface_ms1mv3_r100",
    "ada": "adaface_ms1mv3_r100",
    "mag": "magface_ms1mv2_iresnet100",
    "edge": "edgeface_webface12m_xs_gamma_06",
}
MODEL_WEIGHT_PATHS = {
    "arc": "models/arcface/ms1mv3_r100_backbone.pth",
    "ada": "models/adaface/adaface_ir101_ms1mv3.ckpt",
    "mag": "models/magface/magface_ms1mv2.pth",
    "edge": "models/edgeface/edgeface_xs_gamma_06.pt",
}
# AdaFace MS1MV2 bridge를 쓸 때는 profile/checkpoint를 함께 변경합니다.
MODEL_SMOKE_DEVICE = "cuda"
ARTIFACT_STORAGE_MODE = "results_only"

EXECUTE = True
ACKNOWLEDGE_LOCAL_EXECUTION = True
START_NEW_RUN = True
COMPLETED_RUN_OVERRIDES = {
   # "lfw": "runs/lfw_YYYYMMDD/<explicit-completed-run>",
   # "survface": "runs/survface_YYYYMMDD/<explicit-completed-run>",
   # "rfw_custom": "runs/rfw_custom_YYYYMMDD/<explicit-completed-run>",
   # "tinyface": "runs/tinyface/YYYY/MM/DD/<explicit-completed-run>",
}
RUN_SEARCH_SPACE_REFRESH = True
RUN_FAITHFULNESS = True  # LFW·SurvFace·RFW-Custom에서 실행; TinyFace에는 적용하지 않음
FAITHFULNESS_MAXIMUM_SAMPLES = 10000  # 양의 정수 상한; None이면 제한 없이 전체 후보 사용
RUN_FINAL_REPORT = True
WRITE_FINAL_REPORT = False
OVERWRITE_FINAL_REPORT = False

# RFW-Official 1:1 supplementary 평가. RFW-Custom 1:N과 별도입니다.
RUN_RFW_VERIFICATION = False
RFW_ORIGIN_ARTIFACT_DIR = "results/rfw_step7/origin_embeddings/{model_uid}"
# 동일 물리 RFW population에서 fit한 codec은 Official external-transfer 근거가 아닙니다.
RFW_CODEC_SOURCE_DATASETS = ("lfw", "survface")
RFW_SELECTED_CODEC_FAMILIES = ("pca", "pq")
RFW_SELECTED_CODEC_PROFILES = None
RFW_ALLOW_ORIGIN_ONLY = False
RFW_BOOTSTRAP_REPEATS = 2000
RFW_REUSE_COMPLETED = True


## 2. 프로젝트와 공통 runner 로드

현재 작업 디렉터리의 상위에서 저장소 루트를 찾습니다. 아래 셀은 실험을 시작하거나 가중치를 로드하지 않습니다.


In [2]:
from pathlib import Path
from pprint import pprint
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.pipeline_runner import (
    FULL_DATA_FRACTION,
    prepare_common_model_checkpoint,
)
from research.runtime import ProgressReporter
from research.evaluation import PQ_SEARCH_MODES
from research.experiments import (
    INTEGRATED_DATASET_IDS,
    OPEN_SET_DATASET_IDS,
    build_integrated_experiment_plans,
    evaluate_rfw_frozen_codecs,
    frozen_codec_specs_from_completed_run,
    inspect_integrated_experiment_plans,
    is_open_set_dataset,
    rfw_frozen_codec_evaluation_uid,
    run_or_reuse_integrated_experiment,
    validate_completed_run_overrides,
    validate_integrated_dataset_ids,
    validate_integrated_quick_data_fractions,
)
from scripts.run_integrated_postprocessing import (
    postprocess_completed_run,
    run_cross_dataset_report_notebook,
)

DATASET_IDS = validate_integrated_dataset_ids(DATASET_IDS)
QUICK_DATA_FRACTIONS = validate_integrated_quick_data_fractions(
    QUICK_DATA_FRACTIONS
)
COMPLETED_RUN_OVERRIDES = validate_completed_run_overrides(
    COMPLETED_RUN_OVERRIDES,
    selected_dataset_ids=DATASET_IDS,
)
SELECTED_OPEN_SET_DATASET_IDS = tuple(
    dataset_id for dataset_id in DATASET_IDS if is_open_set_dataset(dataset_id)
)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"RUN_TIER={RUN_TIER}")
print(f"DATASET_IDS={DATASET_IDS}")
print(f"INTEGRATED_DATASET_IDS={INTEGRATED_DATASET_IDS}")
print(f"OPEN_SET_DATASET_IDS={OPEN_SET_DATASET_IDS}")
print(f"QUICK_DATA_FRACTIONS={QUICK_DATA_FRACTIONS}")
print(f"TARGET_FPIRS={TARGET_FPIRS}")

if tuple(PQ_SEARCH_MODES) != EXPECTED_PQ_SEARCH_MODES:
    raise ValueError(
        f"PQ search-condition contract mismatch: {PQ_SEARCH_MODES}"
    )
if not isinstance(RUN_PQ_SDC, bool):
    raise TypeError("RUN_PQ_SDC는 bool이어야 합니다.")
expected_pq_sdc_settings = ((128, 8),) if RUN_PQ_SDC else ()
if tuple(PQ_SDC_SETTINGS) != expected_pq_sdc_settings:
    raise ValueError("PQ_SDC_SETTINGS가 RUN_PQ_SDC 선택과 일치하지 않습니다.")
if FAITHFULNESS_MAXIMUM_SAMPLES is not None and (
    isinstance(FAITHFULNESS_MAXIMUM_SAMPLES, bool)
    or not isinstance(FAITHFULNESS_MAXIMUM_SAMPLES, int)
    or FAITHFULNESS_MAXIMUM_SAMPLES <= 0
):
    raise ValueError("FAITHFULNESS_MAXIMUM_SAMPLES는 None 또는 양의 정수여야 합니다.")
SELECTED_PQ_SEARCH_MODES = tuple(
    mode
    for mode in PQ_SEARCH_MODES
    if RUN_PQ_SDC or mode != "pq_sdc_exhaustive"
)
print(f"FULL_DATA_FRACTION={FULL_DATA_FRACTION}")
print(f"RUN_PQ_SDC={RUN_PQ_SDC}")
print(f"PQ_SDC_SETTINGS={PQ_SDC_SETTINGS}")
print(f"SELECTED_PQ_SEARCH_MODES={SELECTED_PQ_SEARCH_MODES}")
print(f"FAITHFULNESS_MAXIMUM_SAMPLES={FAITHFULNESS_MAXIMUM_SAMPLES}")


PROJECT_ROOT=C:\ronbun
MODEL_NAME=arc
RUN_TIER=full
DATASET_IDS=('lfw', 'survface', 'rfw_custom', 'tinyface')
INTEGRATED_DATASET_IDS=('lfw', 'survface', 'rfw_custom', 'tinyface')
OPEN_SET_DATASET_IDS=('lfw', 'survface', 'rfw_custom')
QUICK_DATA_FRACTIONS={'lfw': 0.1, 'survface': 0.02, 'rfw_custom': 0.1, 'tinyface': 0.1}
TARGET_FPIRS=(0.01, 0.05, 0.1, 0.2, 0.3)
FULL_DATA_FRACTION=1.0
PQ_SEARCH_MODES=('pq_reconstruction_cosine', 'pq_one_sided_cosine', 'pq_adc_exhaustive', 'pq_sdc_exhaustive')


## 3. 선택 모델과 가중치 고정

선택한 별칭에서 profile과 가중치 경로를 가져와 checkpoint SHA-256, 전처리, target layer를 하나의 `model_uid`로 등록합니다. 등록 정보가 같으면 재사용합니다. 기존 smoke 검증 결과가 있으면 재사용하고, 없으면 최대 8장으로 forward/target-layer smoke test를 수행합니다. 전체 데이터 실험은 시작하지 않습니다.


In [3]:
if MODEL_NAME not in MODEL_PROFILE_BY_NAME or MODEL_NAME not in MODEL_WEIGHT_PATHS:
    raise ValueError("MODEL_NAME은 'arc', 'ada', 'mag', 'edge' 중 하나여야 합니다.")

MODEL_PROFILE = MODEL_PROFILE_BY_NAME[MODEL_NAME]
MODEL_WEIGHT_PATH = PROJECT_ROOT / MODEL_WEIGHT_PATHS[MODEL_NAME]

MODEL_PREPARATION = prepare_common_model_checkpoint(
    project_root=PROJECT_ROOT,
    model_name=MODEL_NAME,
    model_profile=MODEL_PROFILE,
    checkpoint_path=MODEL_WEIGHT_PATH,
    run_smoke_validation=True,
    smoke_device=MODEL_SMOKE_DEVICE,
    seed=SEED,
)
pprint(MODEL_PREPARATION.as_dict(), sort_dicts=False)


{'model_name': 'arc',
 'model_profile': 'arcface_ms1mv3_r100',
 'family': 'arcface',
 'checkpoint_path': 'C:\\ronbun\\models\\arcface\\ms1mv3_r100_backbone.pth',
 'checkpoint_sha256': 'a566a62357f0c55b679d9ff2f022a294486568be0c00665d39029d0e46a8109b',
 'model_uid': 'arcface-7972a704552df378345f',
 'model_spec_path': 'C:\\ronbun\\runs\\step2\\model_registry\\arcface-7972a704552df378345f.json',
 'smoke_validation_status': 'validated_now',
 'smoke_validation_path': 'C:\\ronbun\\runs\\step2\\model_validation\\arcface-7972a704552df378345f\\smoke_summary.json'}


## 4. 결정적 실행 plan 생성

manifest를 읽어 실제 선택 예정 행 수와 role/split 분포를 계산합니다. 같은 manifest hash·seed·tier라면 같은 identity 집합을 선택합니다. 이 셀도 DB나 run을 변경하지 않습니다.


In [4]:
PLANS = build_integrated_experiment_plans(
    project_root=PROJECT_ROOT,
    dataset_ids=DATASET_IDS,
    run_tier=RUN_TIER,
    quick_data_fractions=QUICK_DATA_FRACTIONS,
    seed=SEED,
    model_preparation=MODEL_PREPARATION,
    model_name=MODEL_NAME,
    artifact_storage_mode=ARTIFACT_STORAGE_MODE,
    device=MODEL_SMOKE_DEVICE,
    pq_sdc_settings=PQ_SDC_SETTINGS,
)

# 4 models × 3 open-set datasets 통합 표를 만들 때만 12개 완료 run을 명시합니다.
# TinyFace 4개 checkpoint 결과는 각 모델 보고서의 supplementary 표로 유지합니다.
# 자동 latest 선택은 하지 않으며 model/dataset/run lineage가 모두 검증됩니다.
CROSS_MODEL_RUN_MATRIX = {
    # "arcface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
    # "adaface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
    # "magface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
    # "edgeface": {"lfw": "runs/...", "survface": "runs/...", "rfw_custom": "runs/..."},
}
for dataset_id in SELECTED_OPEN_SET_DATASET_IDS:
    plan = PLANS[dataset_id]
    observed_targets = tuple(
        float(value)
        for value in plan.effective_step4_config["evaluation"]["reported_target_fpirs"]
    )
    if observed_targets != TARGET_FPIRS:
        raise ValueError(
            f"{dataset_id}: FPIR target contract mismatch: {observed_targets}"
        )
for dataset_id, plan in PLANS.items():
    print(f"\n[{dataset_id}] plan")
    pprint(plan.as_dict(), sort_dicts=False)



[lfw] plan
{'plan_id': '3482a2c54de7bddd',
 'pipeline_id': 'common_step4_gradcam_v1',
 'dataset_id': 'lfw',
 'run_tier': 'full',
 'data_fraction': 1.0,
 'quick_data_fractions': {'lfw': 0.1,
                          'survface': 0.02,
                          'rfw_custom': 0.1,
                          'tinyface': 0.1},
 'quick_fraction_override': False,
 'seed': 8972,
 'model_name': 'arc',
 'model_profile': 'arcface_ms1mv3_r100',
 'model_uid': 'arcface-7972a704552df378345f',
 'model_checkpoint_path': 'C:\\ronbun\\models\\arcface\\ms1mv3_r100_backbone.pth',
 'pq_sdc_settings': [[128, 8]],
 'evaluation_contract_id': 'face_search_evaluation_v1',
 'evaluation_contract_path': 'C:\\ronbun\\configs\\experiments\\evaluation_contract_v1.yaml',
 'evaluation_contract_sha256': '42a734795bf298382349bac15b93a21fc8075ed11e37a4bbd1c7c63f393e9713',
 'base_step4_config_path': 'C:\\ronbun\\configs\\experiments\\step2_pytorch_gradcam.yaml',
 'source_rows': 13233,
 'selected_source_rows': 13233,
 'sourc

## 5. 로컬 preflight

등록 checkpoint, CUDA, ONNX Runtime provider, canonical aligned/landmark bundle, 로컬 Git source 상태를 읽기 전용으로 검사합니다. GitHub나 원격 CI는 사용하지 않습니다. quick은 dirty source를 허용하되 commit과 diff hash를 plan에 고정하고, full은 clean local commit을 요구합니다. `ready_to_execute_pipeline=False`이면 아래 실행 셀의 오류에 표시되는 실패 항목을 먼저 해결합니다.


In [5]:
PREFLIGHTS = inspect_integrated_experiment_plans(PLANS)
for dataset_id, preflight in PREFLIGHTS.items():
    print(f"\n[{dataset_id}] preflight")
    pprint(preflight, sort_dicts=False)



[lfw] preflight
{'plan': {'plan_id': '3482a2c54de7bddd',
          'pipeline_id': 'common_step4_gradcam_v1',
          'dataset_id': 'lfw',
          'run_tier': 'full',
          'data_fraction': 1.0,
          'quick_data_fractions': {'lfw': 0.1,
                                   'survface': 0.02,
                                   'rfw_custom': 0.1,
                                   'tinyface': 0.1},
          'quick_fraction_override': False,
          'seed': 8972,
          'model_name': 'arc',
          'model_profile': 'arcface_ms1mv3_r100',
          'model_uid': 'arcface-7972a704552df378345f',
          'model_checkpoint_path': 'C:\\ronbun\\models\\arcface\\ms1mv3_r100_backbone.pth',
          'pq_sdc_settings': [[128, 8]],
          'evaluation_contract_id': 'face_search_evaluation_v1',
          'evaluation_contract_path': 'C:\\ronbun\\configs\\experiments\\evaluation_contract_v1.yaml',
          'evaluation_contract_sha256': '42a734795bf298382349bac15b93a21fc8075ed11e37

## 6. 사용자 승인 후 순차 실행 또는 재개

`EXECUTE=True`, `ACKNOWLEDGE_LOCAL_EXECUTION=True`일 때만 실제 실험을 시작합니다. 실행 전에 위 plan과 preflight에서 데이터셋·fraction·model_uid·checkpoint 경로를 확인하십시오.

완료된 phase는 건너뛰고, 실패하거나 아직 실행하지 않은 phase부터 이어갑니다. 장시간 loop 로그는 약 10% 경계에서만 출력됩니다. 같은 plan의 완료 run이 있으면 자동으로 새 run을 만들지 않습니다.


In [6]:
EXECUTION_RESULTS = {}
POSTPROCESS_RESULTS = {}
FINAL_REPORT_RESULT = {"status": "not_started"}
RFW_RESULT = {"status": "not_started"}

if EXECUTE:
    if ACKNOWLEDGE_LOCAL_EXECUTION is not True:
        raise RuntimeError(
            "실제 실행 전 ACKNOWLEDGE_LOCAL_EXECUTION=True가 필요합니다."
        )
    for dataset_id, preflight in PREFLIGHTS.items():
        if preflight["ready_to_execute_pipeline"] is True:
            continue
        checks = preflight.get("readiness", {}).get(
            "checks", preflight.get("checks", {})
        )
        raise RuntimeError(f"{dataset_id} preflight 실패: {checks}")

    for dataset_id, plan in PLANS.items():
        progress = ProgressReporter(
            f"{dataset_id}/{RUN_TIER}/{MODEL_NAME}",
            heartbeat_seconds=None,
            milestone_percent=(5 if dataset_id == "tinyface" else 10),
        )
        override = COMPLETED_RUN_OVERRIDES.get(dataset_id)
        execution = run_or_reuse_integrated_experiment(
            plan,
            execution_acknowledged=True,
            start_new_run=START_NEW_RUN,
            completed_run_override=(PROJECT_ROOT / override if override else None),
            progress=progress.callback(
                key_prefix=f"{dataset_id}:{RUN_TIER}:"
            ),
        )
        EXECUTION_RESULTS[dataset_id] = execution
        if execution.get("status") not in {"completed", "already_completed"}:
            raise RuntimeError(
                f"{dataset_id}: 완료 run을 확보하지 못했습니다: {execution}"
            )

    # protocol-specific 후처리는 dispatcher 외부에서도 명시적으로 분리한다.
    # TinyFace official 결과는 자체 완료 artifact이므로 open-set refresh를 적용하지 않는다.
    for dataset_id in SELECTED_OPEN_SET_DATASET_IDS:
        execution = EXECUTION_RESULTS[dataset_id]
        POSTPROCESS_RESULTS[dataset_id] = postprocess_completed_run(
            execution["run_dir"],
            refresh_search_spaces=RUN_SEARCH_SPACE_REFRESH,
            derive_faithfulness=RUN_FAITHFULNESS,
            faithfulness_options={
                "maximum_samples": FAITHFULNESS_MAXIMUM_SAMPLES,
            },
            target_fpirs=TARGET_FPIRS,
        )
    if "tinyface" in EXECUTION_RESULTS:
        POSTPROCESS_RESULTS["tinyface"] = {
            "status": "not_applicable",
            "reason": "TinyFace official artifact is finalized by its canonical runner.",
        }

    if RUN_RFW_VERIFICATION:
        origin_dir = PROJECT_ROOT / RFW_ORIGIN_ARTIFACT_DIR.format(
            model_uid=MODEL_PREPARATION.model_uid
        )
        codec_specs = []
        for source_dataset in RFW_CODEC_SOURCE_DATASETS:
            if source_dataset not in EXECUTION_RESULTS:
                continue
            codec_specs.extend(
                frozen_codec_specs_from_completed_run(
                    EXECUTION_RESULTS[source_dataset]["run_dir"],
                    expected_model_uid=MODEL_PREPARATION.model_uid,
                    families=RFW_SELECTED_CODEC_FAMILIES,
                    profile_names=RFW_SELECTED_CODEC_PROFILES,
                )
            )
        codec_specs = tuple(codec_specs)
        if not codec_specs and not RFW_ALLOW_ORIGIN_ONLY:
            raise RuntimeError(
                "RFW verification requires frozen codecs from an explicitly "
                "selected completed run, or RFW_ALLOW_ORIGIN_ONLY=True."
            )
        rfw_uid = rfw_frozen_codec_evaluation_uid(
            origin_artifact_dir=origin_dir,
            codec_specs=codec_specs,
            strict_official=True,
            bootstrap_seed=SEED,
            bootstrap_repeats=RFW_BOOTSTRAP_REPEATS,
        )
        rfw_output_dir = (
            PROJECT_ROOT / "results/rfw_step7/frozen_codec_evaluation"
            / MODEL_PREPARATION.model_uid / rfw_uid
        )
        rfw_evaluation = evaluate_rfw_frozen_codecs(
            origin_artifact_dir=origin_dir,
            codec_specs=codec_specs,
            output_dir=rfw_output_dir,
            strict_official=True,
            bootstrap_seed=SEED,
            bootstrap_repeats=RFW_BOOTSTRAP_REPEATS,
            reuse_completed=RFW_REUSE_COMPLETED,
        )
        RFW_RESULT = {
            "status": "completed",
            "evaluation_uid": rfw_uid,
            "output_dir": str(rfw_evaluation.root),
            "profile_rows": len(rfw_evaluation.profile_summary),
            "codec_count": len(codec_specs),
        }

    if RUN_FINAL_REPORT and SELECTED_OPEN_SET_DATASET_IDS:
        FINAL_REPORT_RESULT = run_cross_dataset_report_notebook(
            PROJECT_ROOT,
            model_name=MODEL_NAME,
            selected_runs={
                dataset_id: EXECUTION_RESULTS[dataset_id]["run_dir"]
                for dataset_id in DATASET_IDS
            },
            include_faithfulness=RUN_FAITHFULNESS,
            write_outputs=WRITE_FINAL_REPORT,
            overwrite_outputs=OVERWRITE_FINAL_REPORT,
            faithfulness_maximum_samples=FAITHFULNESS_MAXIMUM_SAMPLES,
            pq_sdc_settings=PQ_SDC_SETTINGS,
            rfw_evaluation_dir=(
                RFW_RESULT["output_dir"]
                if RFW_RESULT.get("status") == "completed"
                else None
            ),
            cross_model_run_matrix=(CROSS_MODEL_RUN_MATRIX or None),
        )
    elif RUN_FINAL_REPORT:
        FINAL_REPORT_RESULT = {
            "status": "not_applicable",
            "reason": (
                "cross-dataset report requires at least one selected open-set run; "
                "the TinyFace completed artifact remains independently reportable."
            ),
        }
else:
    EXECUTION_RESULTS = {
        dataset_id: {
            "status": "not_started",
            "reason": "EXECUTE=False; plan과 preflight만 수행했습니다.",
        }
        for dataset_id in DATASET_IDS
    }

INTEGRATED_RESULT = {
    "execution": EXECUTION_RESULTS,
    "postprocessing": POSTPROCESS_RESULTS,
    "rfw_verification": RFW_RESULT,
    "tinyface_official": EXECUTION_RESULTS.get(
        "tinyface", {"status": "not_selected"}
    ),
    "final_report": FINAL_REPORT_RESULT,
}
pprint(INTEGRATED_RESULT, sort_dicts=False)


[01:45:35] lfw/full/arc | origin embedding extraction | elapsed=9s | progress=10% processed=1344 total=13195 rate=144.65/s eta=1m 22s
[01:45:38] lfw/full/arc | origin embedding extraction | elapsed=13s | progress=20% processed=2688 total=13195 rate=209.60/s eta=50s
[01:45:42] lfw/full/arc | origin embedding extraction | elapsed=16s | progress=30% processed=3968 total=13195 rate=245.62/s eta=38s
[01:45:45] lfw/full/arc | origin embedding extraction | elapsed=20s | progress=40% processed=5312 total=13195 rate=269.24/s eta=29s
[01:45:49] lfw/full/arc | origin embedding extraction | elapsed=23s | progress=50% processed=6656 total=13195 rate=286.47/s eta=23s
[01:45:52] lfw/full/arc | origin embedding extraction | elapsed=27s | progress=60% processed=7936 total=13195 rate=296.21/s eta=18s
[01:45:56] lfw/full/arc | origin embedding extraction | elapsed=30s | progress=70% processed=9280 total=13195 rate=305.22/s eta=13s
[01:45:59] lfw/full/arc | origin embedding extraction | elapsed=34s | prog

C:\ronbun\research\experiments\step4_workflow.py:913: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(selected_path)


[12:58:14] rfw_custom/full/arc | origin embedding extraction | elapsed=25s | progress=10% processed=4096 total=40520 rate=164.41/s eta=3m 42s
[12:58:25] rfw_custom/full/arc | origin embedding extraction | elapsed=36s | progress=20% processed=8128 total=40520 rate=226.63/s eta=2m 23s
[12:58:36] rfw_custom/full/arc | origin embedding extraction | elapsed=47s | progress=30% processed=12160 total=40520 rate=258.21/s eta=1m 50s
[12:58:47] rfw_custom/full/arc | origin embedding extraction | elapsed=58s | progress=40% processed=16256 total=40520 rate=279.22/s eta=1m 27s
[12:58:58] rfw_custom/full/arc | origin embedding extraction | elapsed=1m 09s | progress=50% processed=20288 total=40520 rate=293.17/s eta=1m 09s
[12:59:09] rfw_custom/full/arc | origin embedding extraction | elapsed=1m 20s | progress=60% processed=24320 total=40520 rate=303.27/s eta=53s
[12:59:20] rfw_custom/full/arc | origin embedding extraction | elapsed=1m 31s | progress=70% processed=28416 total=40520 rate=310.86/s eta=39

C:\ronbun\research\experiments\step4_workflow.py:1016: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(selected_path)


[13:00:25] rfw_custom/full/arc | population Grad-CAM extraction | elapsed=2m 36s | progress=10% processed=1000 total=10000 rate=6.41/s eta=23m 25s
[13:00:32] rfw_custom/full/arc | population Grad-CAM extraction | elapsed=2m 43s | progress=20% processed=2000 total=10000 rate=12.31/s eta=10m 50s
[13:00:38] rfw_custom/full/arc | population Grad-CAM extraction | elapsed=2m 49s | progress=30% processed=3000 total=10000 rate=17.74/s eta=6m 35s
[13:00:45] rfw_custom/full/arc | population Grad-CAM extraction | elapsed=2m 56s | progress=40% processed=4000 total=10000 rate=22.78/s eta=4m 23s
[13:00:51] rfw_custom/full/arc | population Grad-CAM extraction | elapsed=3m 02s | progress=50% processed=5000 total=10000 rate=27.48/s eta=3m 02s
[13:00:57] rfw_custom/full/arc | population Grad-CAM extraction | elapsed=3m 08s | progress=60% processed=6000 total=10000 rate=31.84/s eta=2m 06s
[13:01:04] rfw_custom/full/arc | population Grad-CAM extraction | elapsed=3m 15s | progress=70% processed=7000 total=

...

[13:02:08] rfw_custom/full/arc | rfw_custom compressor fit | elapsed=4m 19s | progress=50% processed=5 total=10 rate=0.02/s eta=4m 19s family=pca
[13:02:10] rfw_custom/full/arc | rfw_custom compressor fit | elapsed=4m 21s | progress=60% processed=6 total=10 rate=0.02/s eta=2m 54s family=pq profile=pq_512_m8_b8
[13:02:14] rfw_custom/full/arc | rfw_custom compressor fit | elapsed=4m 25s | progress=70% processed=7 total=10 rate=0.03/s eta=1m 53s family=pq profile=pq_512_m16_b8
[13:02:20] rfw_custom/full/arc | rfw_custom compressor fit | elapsed=4m 31s | progress=80% processed=8 total=10 rate=0.03/s eta=1m 08s family=pq profile=pq_512_m32_b8
[13:02:32] rfw_custom/full/arc | rfw_custom compressor fit | elapsed=4m 43s | progress=90% processed=9 total=10 rate=0.03/s eta=31s family=pq profile=pq_512_m64_b8
[13:02:44] rfw_custom/full/arc | rfw_custom compressor fit | elapsed=4m 55s | progress=100% processed=10 total=10 rate=0.03/s eta=0s family=pq profile=pq_512_m128_b8
[13:03:04] rfw_custom/fu

C:\ronbun\research\experiments\step4_workflow.py:2309: DtypeWarning: Columns (0: is_mated) have mixed types. Specify dtype option on import or set low_memory=False.
  selected = pd.read_csv(selected_path)


[13:29:12] tinyface/full/arc | TinyFace embedding extraction | elapsed=2m 05s | progress=5% processed=8704 total=169403 rate=69.91/s eta=38m 19s phase=tinyface_embedding_extraction fraction=0.0513804360017237
[13:30:05] tinyface/full/arc | TinyFace embedding extraction | elapsed=2m 58s | progress=10% processed=17152 total=169403 rate=96.49/s eta=26m 18s phase=tinyface_embedding_extraction fraction=0.10124968270927906
[13:31:01] tinyface/full/arc | TinyFace embedding extraction | elapsed=3m 54s | progress=15% processed=25600 total=169403 rate=109.43/s eta=21m 54s phase=tinyface_embedding_extraction fraction=0.1511189294168344
[13:31:58] tinyface/full/arc | TinyFace embedding extraction | elapsed=4m 51s | progress=20% processed=34048 total=169403 rate=117.07/s eta=19m 16s phase=tinyface_embedding_extraction fraction=0.20098817612438977
[13:32:55] tinyface/full/arc | TinyFace embedding extraction | elapsed=5m 48s | progress=25% processed=42496 total=169403 rate=122.13/s eta=17m 19s phase=

C:\ronbun\research\evaluation\tinyface.py:399: DtypeWarning: Columns (0: native_rank_1_success, 1: decoded_native_rank_1_mismatch, 2: native_rank_5_success, 3: decoded_native_rank_5_mismatch, 4: native_rank_10_success, 5: decoded_native_rank_10_mismatch, 6: native_rank_20_success, 7: decoded_native_rank_20_mismatch) have mixed types. Specify dtype option on import or set low_memory=False.
  per_query = pd.read_csv(validated["per_query.csv"])


CellExecutionError: An error occurred while executing the following cell:
------------------
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if all((candidate / name).exists() for name in ("research", "configs", "runs")):
            return candidate
    raise FileNotFoundError("C:/ronbun 내부에서 노트북을 실행하십시오.")


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.evaluation import validate_rfw_custom_calibration_contract

# 사용자 선택 영역: 모델·데이터셋·완료 run만 바꾸고 아래 로직은 수정하지 않습니다.
# 직접 실행은 매번 아래 기본값으로 초기화하여 이전 커널 상태를 재사용하지 않습니다.
# 배치 runner가 삽입한 parameter cell이 명시적 sentinel을 제공한 경우에만 외부 선택을 

## 7. 결과 해석 경계

- `full`이 기본이며 선택된 각 데이터셋에서 100%를 사용합니다. `QUICK_DATA_FRACTIONS`는 `RUN_TIER="quick"`일 때만 적용됩니다.
- 네 데이터셋은 동일한 `DATASET_IDS`, quick fraction, completed-run override 계약을 사용하지만 평가 protocol은 내부 dispatcher에서 분리됩니다.
- `full`끼리도 model UID, preprocessing, protocol, calibration, codec lineage가 같을 때만 직접 비교합니다.
- ArcFace·AdaFace·MagFace·EdgeFace 비교 단위는 선택한 pretrained checkpoint입니다. loss 함수 자체의 인과적 우월성으로 해석하지 않습니다.
- RFW-Custom은 비공식 identity-disjoint 1:N DIR/FPIR이며 RFW-Official의 pair/fold를 사용하지 않습니다.
- RFW-Official은 1:1 TAR/FAR/EER supplementary 결과이며 open-set 표와 결합하지 않습니다.
- TinyFace는 공식 closed-set distractor protocol(mAP, Rank-1/5/10/20)로 별도 보고합니다. non-mated probe가 없으므로 FPIR/TPIR 또는 threshold calibration 행을 만들지 않습니다.
- CI는 probe-level Wilson 및 paired bootstrap입니다. identity-cluster 또는 checkpoint 재학습 불확실성을 뜻하지 않습니다.
- PQ reconstruction cosine과 exhaustive ADC는 별도 search mode입니다. ADC를 IVF-PQ 또는 pgvector ANN latency로 해석하지 않습니다.
- v6 PQ 비교는 query/gallery 표현과 distance function을 별도 열로 기록합니다.
- `pq_adc_exhaustive`와 `pq_sdc_exhaustive`에는 `frozen_origin` 행이 존재하면 안 됩니다.
- `RUN_PQ_SDC=True`일 때만 SDC를 PQ-m128 `(m=128, nbits=8)`에서 수행합니다. `False`이면 네 데이터셋 모두에서 SDC를 제외하며 주 PQ 비교에는 영향을 주지 않습니다.
- `FAITHFULNESS_MAXIMUM_SAMPLES`는 LFW·SurvFace·RFW-Custom 각각의 faithfulness 표본 상한입니다. `None`이면 제한 없이 전체 후보를 사용하며 TinyFace에는 적용하지 않습니다.
- Grad-CAM faithfulness는 검증된 high/low/random control artifact가 있을 때만 포함합니다.
